# Final Project Environmental Factors — Solar Irradiance (DNI)
**Business 35137 — Winter 2026**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
DATA_DIR   = '../Data/'
OUT_DIR    = os.path.join(DATA_DIR, 'Intermediate/')
os.makedirs(OUT_DIR, exist_ok=True)

SOLAR_FILE = os.path.join(DATA_DIR, 'nsrdb solar data.csv')

In [ ]:
# ── Load Raw Solar Data ────────────────────────────────────────────────────────
# Source: NSRDB (National Solar Radiation Database)
# Granularity: state-level monthly averages
# DNI = Direct Normal Irradiance (W/m²)

solar = pd.read_csv(SOLAR_FILE)

# Standardise column names
solar = solar.rename(columns={'Year': 'year', 'Month': 'month_num'})

# Map month number -> abbreviated month name to mirror the weather panel
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_name_map = {i+1: m for i, m in enumerate(MONTHS)}
solar['month'] = solar['month_num'].map(month_name_map)

print(f"Solar data: {len(solar):,} rows")
print(f"States    : {solar['State_Abbr'].nunique()}")
print(f"Year range: {solar['year'].min()} – {solar['year'].max()}")
solar.head()

In [ ]:
# ── Create State-Month Identifier ─────────────────────────────────────────────
# Mirrors county_month in the weather panel, but at state level.
# e.g. 'TX_Jul' = Texas, July — compared across all years in the dataset.

solar['state_month'] = solar['State_Abbr'] + '_' + solar['month']

print('Unique state-month groups :', solar['state_month'].nunique())
print('Avg observations per group:', round(
    solar['DNI'].notna().sum() / solar['state_month'].nunique(), 1
))
solar[['State_Abbr','year','month','state_month']].head(8)

In [ ]:
# ── Compute Z-Scores and Directional Anomaly Flags (DNI) ──────────────────────
# For each (state, month) pair:
#   sm_mean = mean of all non-missing DNI values across all years
#   sm_std  = std  of all non-missing DNI values across all years
#   z_score = (DNI - sm_mean) / sm_std
#   anomaly = 'above' if z > +2  |  'below' if z < -2  |  None if normal

stats = (
    solar.dropna(subset=['DNI'])
         .groupby('state_month')['DNI']
         .agg(DNI_sm_mean='mean', DNI_sm_std='std')
         .reset_index()
)
solar = solar.merge(stats, on='state_month', how='left')
solar['DNI_z'] = (solar['DNI'] - solar['DNI_sm_mean']) / solar['DNI_sm_std']
solar['DNI_anomaly'] = np.where(
    solar['DNI_z'] >  2, 'above',
    np.where(solar['DNI_z'] < -2, 'below', None)
)

above = (solar['DNI_anomaly'] == 'above').sum()
below = (solar['DNI_anomaly'] == 'below').sum()
total = solar['DNI'].notna().sum()
print(f"DNI: {above:,} above ({above/total*100:.2f}%)  |  {below:,} below ({below/total*100:.2f}%)")

solar[['State_Abbr','year','month','state_month','DNI','DNI_sm_mean','DNI_sm_std','DNI_z','DNI_anomaly']].head(8)

In [ ]:
# ── Sanity Check: Z-Score Distribution ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

z = solar['DNI_z'].dropna()
flag_pct = (solar['DNI_anomaly'].isin(['above','below'])).mean() * 100
ax.hist(z, bins=80, color='goldenrod', alpha=0.75, edgecolor='none')
ax.axvline(-2, color='red', linestyle='--', linewidth=1.2, label='±2 std threshold')
ax.axvline( 2, color='red', linestyle='--', linewidth=1.2)
ax.set_title(f'DNI Z-Score Distribution\n({flag_pct:.1f}% flagged as anomalies)')
ax.set_xlabel('Z-Score')
ax.set_ylabel('Count')
ax.legend()

plt.suptitle('State-Month Normalized Solar DNI Anomaly Detection (NSRDB)', y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Build Final Panel & Save Outputs ──────────────────────────────────────────
# Column order mirrors the weather_panel.parquet convention.

panel_cols = [
    'State', 'State_Abbr', 'year', 'month_num', 'month', 'state_month',
    'DNI', 'DNI_sm_mean', 'DNI_sm_std', 'DNI_z', 'DNI_anomaly',
]

panel = solar[panel_cols].sort_values(['State_Abbr','year','month_num']).reset_index(drop=True)

print(f"Solar panel: {len(panel):,} rows  |  {panel['State_Abbr'].nunique()} states")
print(f"Columns    : {list(panel.columns)}")
panel.head(6)

In [ ]:
# ── Save to Parquet ────────────────────────────────────────────────────────────
solar_out = os.path.join(OUT_DIR, 'solar_panel.parquet')
panel.to_parquet(solar_out, index=False)

print(f"Saved -> {solar_out}")
print("\nsolar_panel.parquet columns:")
print(list(pd.read_parquet(solar_out).columns))